In [3]:
import pandas as pd
import numpy as np
import re
from collections import Counter
import math
from io import StringIO

# DNS logs data
dns_data = """timestamp,client_ip,qtype,domain,resolver_ip
2026-02-28T20:00:01,192.168.1.10,A,google.com,8.8.8.8
2026-02-28T20:00:02,192.168.1.50,A,paypal.com,1.1.1.1
2026-02-28T20:00:03,192.168.1.50,TXT,x7k9p2m4q8r5t1n3v6b9.attacker.com,1.1.1.1
2026-02-28T20:00:04,192.168.1.50,TXT,a2f8d4j7k9m3p6q1r5t8v2.attacker.com,1.1.1.1
2026-02-28T20:00:05,192.168.1.20,A,example.com,8.8.8.8
2026-02-28T20:00:06,192.168.1.50,TXT,z9w3v6b2n5m8k1p4r7t0.attacker.com,1.1.1.1
2026-02-28T20:00:07,192.168.1.50,TXT,k3n7q1r4t8v2x6c9z0.attacker.com,1.1.1.1
"""

# Load and prepare data
df_dns = pd.read_csv(StringIO(dns_data))
df_dns['timestamp'] = pd.to_datetime(df_dns['timestamp'])
df_dns['subdomain'] = df_dns['domain'].str.split('.').str[0]

# Entropy function
def shannon_entropy(text):
    if len(text) == 0:
        return 0.0
    counter = Counter(text.lower())
    entropy = 0
    length = len(text)
    for count in counter.values():
        p = count / length
        if p > 0:
            entropy -= p * math.log2(p)
    return entropy

# Risk scoring function
def calculate_risk(row):
    score = 0
    if row['entropy'] > 3.5:
        score += 40
    elif row['entropy'] > 3.0:
        score += 25
    if row['qtype'] == 'TXT':
        score += 30
    if len(row['subdomain']) > 12:
        score += 20
    return min(100, score)

# Calculate all metrics
df_dns['entropy'] = df_dns['subdomain'].apply(shannon_entropy)
df_dns['risk_score'] = df_dns.apply(calculate_risk, axis=1)

# Analysis variables
high_entropy = df_dns[df_dns['entropy'] > 3.0]
txt_queries = df_dns[df_dns['qtype'] == 'TXT']
tunnels = df_dns[df_dns['risk_score'] > 60]
txt_summary = txt_queries.groupby('client_ip').size().reset_index(name='txt_queries')
suspicious_txt = txt_summary[txt_summary['txt_queries'] > 2]

# FINAL REPORT
print("DNS TUNNEL DETECTION REPORT")
print("=" * 50)
print("Total DNS queries analyzed:", len(df_dns))
print("High entropy domains:", len(high_entropy))
print("TXT queries:", len(txt_queries))
print("Suspicious TXT clients:", len(suspicious_txt))
print("Confirmed tunnels (score > 60):", len(tunnels))
print()

if len(high_entropy) > 0:
    print("HIGH ENTROPY DOMAINS:")
    print(high_entropy[['client_ip', 'domain', 'entropy']].sort_values('entropy', ascending=False).to_string(index=False))
    print()

if len(suspicious_txt) > 0:
    print("TXT ABUSE:")
    print(suspicious_txt.to_string(index=False))
    print()

if len(tunnels) > 0:
    print("CONFIRMED DNS TUNNELS:")
    print(tunnels[['client_ip', 'domain', 'qtype', 'entropy', 'risk_score']].sort_values('risk_score', ascending=False).to_string(index=False))
else:
    print("No DNS tunneling detected")

print("\nAnalysis complete - Production ready")

DNS TUNNEL DETECTION REPORT
Total DNS queries analyzed: 7
High entropy domains: 4
TXT queries: 4
Suspicious TXT clients: 1
Confirmed tunnels (score > 60): 4

HIGH ENTROPY DOMAINS:
   client_ip                              domain  entropy
192.168.1.50   z9w3v6b2n5m8k1p4r7t0.attacker.com 4.321928
192.168.1.50 a2f8d4j7k9m3p6q1r5t8v2.attacker.com 4.277613
192.168.1.50   x7k9p2m4q8r5t1n3v6b9.attacker.com 4.221928
192.168.1.50     k3n7q1r4t8v2x6c9z0.attacker.com 4.169925

TXT ABUSE:
   client_ip  txt_queries
192.168.1.50            4

CONFIRMED DNS TUNNELS:
   client_ip                              domain qtype  entropy  risk_score
192.168.1.50   x7k9p2m4q8r5t1n3v6b9.attacker.com   TXT 4.221928          90
192.168.1.50 a2f8d4j7k9m3p6q1r5t8v2.attacker.com   TXT 4.277613          90
192.168.1.50   z9w3v6b2n5m8k1p4r7t0.attacker.com   TXT 4.321928          90
192.168.1.50     k3n7q1r4t8v2x6c9z0.attacker.com   TXT 4.169925          90

Analysis complete - Production ready
